# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: binary classification.** The model predicts an *observed* outcome for each `(content_hash_id, report_date)` observation: whether the page's organic search performance recovers within a defined future window (see Section 2 for the exact label rule).

**Decision this serves:** an SEO Specialist runs a daily review queue and needs to know which declining pages are worth reviewing first. "Will this recover?" is the underlying question, but the specialist doesn't act on a single yes/no — they act on a *ranking*.

**Why classification, not "ranking" as the model type:** ranking here is a **consumption pattern**, not a separate model architecture. The model is trained as a standard binary classifier against an observed 0/1 label. The product layer then takes the model's raw predicted probability — *not* the thresholded label — and sorts pages by it to build the queue. The classification threshold (0.5, or wherever it lands) is a downstream business decision; it has no role in how the queue is built. This separation matters because the ML task and the product behavior are evaluated by different metrics — see Section 3.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Production label definition** (first-principles design, not inherited from any existing `trend_direction`-style field):

For each page, at each observation date `d`:

- **Baseline** = mean `gsc_impressions` over the 30 days *before* `d`
- **Future** = mean `gsc_impressions` over the 30 days *after* `d`
- **Label = 1 (recovered)** if `Future ≥ 0.90 × Baseline`, else **0**

**Why impressions, over clicks / CTR / position:** impressions are the most direct measure of organic *visibility*. CTR conflates visibility with presentation (title/snippet changes can move CTR with no change in whether Google is showing the page). Average position can improve while impressions stay flat if search demand for the underlying queries drops — position doesn't tell you whether anyone is actually seeing the page. Clicks fold in user behavior on top of visibility. Impressions isolate the one signal that answers "is Google still showing this page to people."

**Baseline philosophy — self-referential, not cohort-relative:** the label compares a page to *its own* history, not to a peer group. This keeps the label directly observable, auditable, and free of an arbitrary cohort-definition dispute. Cohort-relative performance is useful, but it's deliberately reserved as an engineered feature instead of being built into the target definition.

**Window and threshold — starting hypotheses, not proven constants:**

- **30-day windows** align with how SEO Specialists work and reduce sensitivity to short-term fluctuations.
- **90% threshold** defines recovery as returning close to the previous baseline while allowing normal variation.

Both values are hypotheses that should be validated during EDA (ML-06).

**Known open questions (deferred):**

- `search_volume` availability (ML-04)
- Page-aware train/test split (ML-05)

**Important limitation:** this label cannot be computed from the starter CSV because it contains only one snapshot per page and no `report_date`. The rolling-window implementation is therefore documented but intentionally not executed in this notebook.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Two metrics, deliberately different, because they answer different questions.

**Development metric — PR-AUC**

If the recovered class is imbalanced (to be confirmed during EDA), PR-AUC is more informative than ROC-AUC because it focuses on the positive class.

**Product metric — Precision@K**

The SEO Specialist only reviews the highest-ranked pages, so the quality of the top recommendations matters more than average model performance.

PR-AUC measures model quality during development, while Precision@K measures whether the final ranked queue is actually useful.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Production unit of analysis:** `(content_hash_id, report_date)` — one row per page, per observation date.

The label is defined for a page at a specific date, so the same page may appear multiple times with different labels across time.

**What's actually loaded below is the starter CSV.**

This week's dataset is different from the intended production grain.

1. The CSV contains **one row per page** (`content_id`) rather than one row per `(content_hash_id, report_date)`.

2. The CSV has no `report_date` column.

The file already includes `impressions_last_30d` and `impressions_prev_30d`, showing that the same rolling-window concept already exists as precomputed snapshots.

**Target column**

The notebook sketches what the future `recovered` column will look like but does not compute it because the required daily observations are unavailable in the starter dataset.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The starter dataset already contains a rule-based system (`trend_direction` and `trend_pct`), but it has important limitations.

### 1. The same "down" trend does not mean the same thing everywhere.

The distribution of `trend_pct` differs across `content_type` and `competition_level`, so one fixed threshold cannot work equally well for every segment.

### 2. Recovery depends on interactions between multiple features.

Whether a page eventually recovers likely depends on content type, competition level, search volume, content freshness, and content age together—not on any single feature in isolation.

Machine learning can learn these interactions directly from historical data, while a manually written rule would require constant tuning and maintenance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.